# 3D reporter timelapse — 00_manifest_qc

**Feeds:** Fig 5h

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 00 | Raw Acquisition Manifest And QC

This notebook inventories the raw FOXF1 / BMP4 3D organoid timelapse export,
writes canonical manifest tables, and records acquisition-level QC before any
downstream biological analysis.


## Cell Guide

1. Discover the active raw dataset under `data/raw/`.
2. Parse per-position `metadata.txt` files and stage-position metadata.
3. Inventory all TIFF frames, including nested-layout exceptions such as `Pos52`.
4. Write position-level and file-level manifests plus QC summary outputs.
5. Review flagged positions and save basic acquisition QC plots.


In [ ]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


In [ ]:
# -------------------------------
# User configuration
# -------------------------------
cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "scripts").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

RAW_ROOT = ROOT / "data/raw"
POS_RE = re.compile(r"Pos(?P<index>\d+)$")


def discover_dataset_dirs(raw_root: Path) -> list[Path]:
    dataset_dirs = []
    for path in sorted(raw_root.rglob("*")):
        if not path.is_dir():
            continue
        try:
            child_dirs = [child for child in path.iterdir() if child.is_dir()]
        except PermissionError:
            continue
        if any(POS_RE.fullmatch(child.name) for child in child_dirs):
            dataset_dirs.append(path)
    return dataset_dirs


DATASET_DIRS = discover_dataset_dirs(RAW_ROOT)
if not DATASET_DIRS:
    raise FileNotFoundError(f"No dataset directories with Pos* folders found under {RAW_ROOT}")

DATASET_DIR = DATASET_DIRS[0]

POSITION_MANIFEST_PATH = ROOT / "results/manifests/acquisition_position_manifest.tsv"
FILE_MANIFEST_PATH = ROOT / "results/manifests/acquisition_file_manifest.tsv"
CHANNEL_QC_PATH = ROOT / "results/qc/frame_counts_by_position_channel.tsv"
SUMMARY_JSON_PATH = ROOT / "results/qc/acquisition_qc_summary.json"
SUMMARY_TXT_PATH = ROOT / "results/qc/acquisition_qc_summary.txt"
FRAME_COUNT_PLOT_PATH = ROOT / "results/qc/frame_counts_per_position.png"
STAGE_MAP_PLOT_PATH = ROOT / "results/qc/stage_map_frame_counts.png"

WRITE_OUTPUTS = True
WRITE_FILE_MANIFEST = True
SAVE_PLOTS = True

print("Project root:", ROOT)
print("Raw root:", RAW_ROOT)
print("Selected dataset:", DATASET_DIR)
if len(DATASET_DIRS) > 1:
    print("Additional dataset candidates detected:")
    for path in DATASET_DIRS[1:]:
        print(" -", path)


## Output Files

This notebook writes:

- `results/manifests/acquisition_position_manifest.tsv`
- `results/manifests/acquisition_file_manifest.tsv`
- `results/qc/frame_counts_by_position_channel.tsv`
- `results/qc/acquisition_qc_summary.json`
- `results/qc/acquisition_qc_summary.txt`
- `results/qc/frame_counts_per_position.png`
- `results/qc/stage_map_frame_counts.png`


In [ ]:
# -------------------------------
# Helpers
# -------------------------------
IMG_RE = re.compile(
    r"img_channel(?P<channel>\d+)_position(?P<position>\d+)_time(?P<time>\d+)_z(?P<z>\d+)\.tif$",
    re.IGNORECASE,
)


def pos_index_from_name(name: str) -> int:
    match = POS_RE.fullmatch(name)
    if not match:
        raise ValueError(f"Unexpected position label: {name}")
    return int(match.group("index"))


def rel_to_root(path: Path) -> str:
    return path.relative_to(ROOT).as_posix()


def load_summary(metadata_path: Path) -> dict:
    return json.loads(metadata_path.read_text())["Summary"]


def stage_lookup_from_summary(summary: dict) -> dict[str, dict]:
    lookup = {}
    for stage_pos in summary.get("StagePositions", []):
        xy = [None, None]
        z_value = None
        for item in stage_pos.get("DevicePositions", []):
            device = item.get("Device")
            values = item.get("Position_um", [])
            if device == "XYStage":
                if len(values) >= 2:
                    xy = [values[0], values[1]]
            elif values:
                z_value = values[0]
        lookup[stage_pos["Label"]] = {
            "grid_row": stage_pos.get("GridRow"),
            "grid_col": stage_pos.get("GridCol"),
            "stage_x_um": xy[0],
            "stage_y_um": xy[1],
            "stage_z_um": z_value,
        }
    return lookup


def parse_image_record(path: Path, pos_dir: Path, channel_names: list[str]) -> dict | None:
    match = IMG_RE.match(path.name)
    if not match:
        return None
    channel_index = int(match.group("channel"))
    parent_rel = path.parent.relative_to(pos_dir)
    storage_subdir = "." if str(parent_rel) == "." else parent_rel.as_posix()
    return {
        "dataset_dir": rel_to_root(DATASET_DIR),
        "position_label": pos_dir.name,
        "position_index": pos_index_from_name(pos_dir.name),
        "channel_index": channel_index,
        "channel_name": channel_names[channel_index] if channel_index < len(channel_names) else f"channel{channel_index:03d}",
        "time_index": int(match.group("time")),
        "z_index": int(match.group("z")),
        "storage_subdir": storage_subdir,
        "relative_path": rel_to_root(path),
    }


def summarize_position(
    pos_dir: Path,
    channel_names: list[str],
    stage_lookup: dict[str, dict],
    intended_frames: int,
    intended_channels: int,
) -> tuple[dict, list[dict], list[dict]]:
    metadata_path = pos_dir / "metadata.txt"
    summary = load_summary(metadata_path)
    image_paths = sorted(pos_dir.rglob("img_channel*_position*_time*_z*.tif"))
    avi_paths = sorted(pos_dir.rglob("*.avi"))

    file_rows = []
    channel_timepoints = defaultdict(set)
    channel_file_counts = Counter()
    storage_subdirs = set()

    for image_path in image_paths:
        record = parse_image_record(image_path, pos_dir, channel_names)
        if record is None:
            continue
        file_rows.append(record)
        channel_file_counts[record["channel_index"]] += 1
        channel_timepoints[record["channel_index"]].add(record["time_index"])
        storage_subdirs.add(record["storage_subdir"])

    channel_rows = []
    for channel_index in sorted(channel_timepoints):
        times = sorted(channel_timepoints[channel_index])
        channel_rows.append(
            {
                "dataset_dir": rel_to_root(DATASET_DIR),
                "position_label": pos_dir.name,
                "position_index": pos_index_from_name(pos_dir.name),
                "channel_index": channel_index,
                "channel_name": channel_names[channel_index] if channel_index < len(channel_names) else f"channel{channel_index:03d}",
                "observed_files": channel_file_counts[channel_index],
                "observed_frames": len(times),
                "time_index_min": times[0] if times else None,
                "time_index_max": times[-1] if times else None,
                "is_contiguous": bool(times) and times == list(range(times[0], times[-1] + 1)),
            }
        )

    frame_counts = sorted({row["observed_frames"] for row in channel_rows})
    observed_frames_per_channel = frame_counts[0] if len(frame_counts) == 1 and frame_counts else None
    frame_index_min = min((row["time_index_min"] for row in channel_rows if row["time_index_min"] is not None), default=None)
    frame_index_max = max((row["time_index_max"] for row in channel_rows if row["time_index_max"] is not None), default=None)
    stage_info = stage_lookup.get(pos_dir.name, {})

    if storage_subdirs == {"."}:
        layout_mode = "flat"
    elif storage_subdirs and all("/" not in subdir and subdir != "." for subdir in storage_subdirs):
        layout_mode = "nested_by_channel"
    elif storage_subdirs:
        layout_mode = "mixed_nested"
    else:
        layout_mode = "no_images"

    record = {
        "dataset_dir": rel_to_root(DATASET_DIR),
        "position_label": pos_dir.name,
        "position_index": pos_index_from_name(pos_dir.name),
        "position_relative_path": rel_to_root(pos_dir),
        "metadata_relative_path": rel_to_root(metadata_path),
        "image_layout": layout_mode,
        "storage_subdirs": ";".join(sorted(storage_subdirs)) if storage_subdirs else "",
        "avi_count": len(avi_paths),
        "avi_present": len(avi_paths) > 0,
        "observed_tiff_count": len(file_rows),
        "observed_channel_count": len(channel_rows),
        "observed_frames_per_channel": observed_frames_per_channel,
        "frame_index_min": frame_index_min,
        "frame_index_max": frame_index_max,
        "all_channels_same_frame_count": len(frame_counts) <= 1,
        "all_channels_contiguous": all(row["is_contiguous"] for row in channel_rows) if channel_rows else False,
        "metadata_declared_frames": summary.get("Frames"),
        "metadata_declared_positions": summary.get("Positions"),
        "metadata_declared_channels": summary.get("Channels"),
        "interval_ms": summary.get("Interval_ms"),
        "start_time": summary.get("StartTime"),
        "intended_frames": intended_frames,
        "intended_channels": intended_channels,
        "missing_frames_per_channel": (
            intended_frames - observed_frames_per_channel
            if observed_frames_per_channel is not None
            else None
        ),
        "missing_files_vs_metadata": intended_frames * intended_channels - len(file_rows),
        "grid_row": stage_info.get("grid_row"),
        "grid_col": stage_info.get("grid_col"),
        "stage_x_um": stage_info.get("stage_x_um"),
        "stage_y_um": stage_info.get("stage_y_um"),
        "stage_z_um": stage_info.get("stage_z_um"),
    }
    return record, channel_rows, file_rows


In [ ]:
# -------------------------------
# Inventory raw acquisition
# -------------------------------
position_dirs = sorted(
    [path for path in DATASET_DIR.iterdir() if path.is_dir() and POS_RE.fullmatch(path.name)],
    key=lambda path: pos_index_from_name(path.name),
)

if not position_dirs:
    raise RuntimeError(f"No Pos* directories found in {DATASET_DIR}")

metadata_available = all((pos_dir / "metadata.txt").exists() for pos_dir in position_dirs)

if metadata_available:
    reference_summary = load_summary(position_dirs[0] / "metadata.txt")
    channel_names = list(reference_summary.get("ChNames", []))
    intended = reference_summary.get("IntendedDimensions", {})
    intended_frames = int(intended.get("time", reference_summary.get("Frames", 0)))
    intended_channels = int(intended.get("channel", reference_summary.get("Channels", 0)))
    stage_lookup = stage_lookup_from_summary(reference_summary)

    position_rows = []
    channel_rows = []
    file_rows = []

    for pos_dir in position_dirs:
        position_record, pos_channel_rows, pos_file_rows = summarize_position(
            pos_dir=pos_dir,
            channel_names=channel_names,
            stage_lookup=stage_lookup,
            intended_frames=intended_frames,
            intended_channels=intended_channels,
        )
        position_rows.append(position_record)
        channel_rows.extend(pos_channel_rows)
        file_rows.extend(pos_file_rows)

    positions_df = pd.DataFrame(position_rows).sort_values("position_index").reset_index(drop=True)
    channel_df = pd.DataFrame(channel_rows).sort_values(["position_index", "channel_index"]).reset_index(drop=True)
    files_df = pd.DataFrame(file_rows).sort_values(["position_index", "channel_index", "time_index"]).reset_index(drop=True)

    common_layout = positions_df["image_layout"].mode().iat[0]
    common_observed_frames = positions_df["observed_frames_per_channel"].mode().iat[0]

    positions_df["is_layout_outlier"] = positions_df["image_layout"] != common_layout
    positions_df["is_frame_count_outlier"] = positions_df["observed_frames_per_channel"] != common_observed_frames
    positions_df["matches_metadata_frame_count"] = positions_df["observed_frames_per_channel"] == positions_df["metadata_declared_frames"]

    dataset_summary = {
        "dataset_dir": rel_to_root(DATASET_DIR),
        "position_count": int(len(position_dirs)),
        "channel_names": channel_names,
        "metadata_declared_frames": int(reference_summary.get("Frames", 0)),
        "metadata_declared_positions": int(reference_summary.get("Positions", 0)),
        "metadata_declared_channels": int(reference_summary.get("Channels", 0)),
        "interval_ms": float(reference_summary.get("Interval_ms", 0)),
        "interval_minutes": round(float(reference_summary.get("Interval_ms", 0)) / 60000.0, 3),
        "start_time": reference_summary.get("StartTime"),
        "observed_frame_count_values": sorted(int(value) for value in positions_df["observed_frames_per_channel"].dropna().unique()),
        "layout_modes": {key: int(value) for key, value in positions_df["image_layout"].value_counts().to_dict().items()},
        "positions_with_avi": positions_df.loc[positions_df["avi_present"], "position_label"].tolist(),
        "positions_with_nonflat_layout": positions_df.loc[positions_df["is_layout_outlier"], "position_label"].tolist(),
        "positions_mismatching_metadata_frames": positions_df.loc[~positions_df["matches_metadata_frame_count"], "position_label"].tolist(),
        "total_tiff_files": int(len(files_df)),
    }

    summary_lines = [
        f"Dataset: {dataset_summary['dataset_dir']}",
        f"Positions found: {dataset_summary['position_count']}",
        f"Channels: {', '.join(channel_names)}",
        f"Metadata declared frames per channel: {dataset_summary['metadata_declared_frames']}",
        f"Observed frame-count values per channel: {dataset_summary['observed_frame_count_values']}",
        f"Layout modes: {dataset_summary['layout_modes']}",
        f"Positions with AVI exports: {dataset_summary['positions_with_avi']}",
        f"Positions with non-flat layout: {dataset_summary['positions_with_nonflat_layout']}",
        f"Positions mismatching metadata-declared frame count: {len(dataset_summary['positions_mismatching_metadata_frames'])}",
        f"Total TIFF files inventoried: {dataset_summary['total_tiff_files']}",
    ]
else:
    required_cached_paths = [
        POSITION_MANIFEST_PATH,
        CHANNEL_QC_PATH,
        SUMMARY_JSON_PATH,
    ]
    missing_cached_paths = [path for path in required_cached_paths if not path.exists()]
    if missing_cached_paths:
        raise FileNotFoundError(
            "Raw per-position metadata.txt files are absent and cached 00-manifest outputs are missing: "
            + ", ".join(str(path) for path in missing_cached_paths)
        )

    positions_df = pd.read_csv(POSITION_MANIFEST_PATH, sep="\t")
    channel_df = pd.read_csv(CHANNEL_QC_PATH, sep="\t")
    if FILE_MANIFEST_PATH.exists():
        files_df = pd.read_csv(FILE_MANIFEST_PATH, sep="\t")
    else:
        files_df = pd.DataFrame(columns=["position_label", "channel_index", "time_index"])
        WRITE_FILE_MANIFEST = False
    dataset_summary = json.loads(SUMMARY_JSON_PATH.read_text())
    channel_names = list(dataset_summary.get("channel_names", []))
    if SUMMARY_TXT_PATH.exists():
        summary_lines = SUMMARY_TXT_PATH.read_text().splitlines()
    else:
        summary_lines = [
            f"Dataset: {dataset_summary['dataset_dir']}",
            f"Positions found: {dataset_summary['position_count']}",
            f"Channels: {', '.join(channel_names)}",
            f"Metadata declared frames per channel: {dataset_summary['metadata_declared_frames']}",
            f"Observed frame-count values per channel: {dataset_summary['observed_frame_count_values']}",
            f"Layout modes: {dataset_summary['layout_modes']}",
            f"Positions with AVI exports: {dataset_summary['positions_with_avi']}",
            f"Positions with non-flat layout: {dataset_summary['positions_with_nonflat_layout']}",
            f"Positions mismatching metadata-declared frame count: {len(dataset_summary['positions_mismatching_metadata_frames'])}",
            f"Total TIFF files inventoried: {dataset_summary['total_tiff_files']}",
        ]

print("\n".join(summary_lines))


In [ ]:
# -------------------------------
# Write outputs
# -------------------------------
output_paths = [
    POSITION_MANIFEST_PATH,
    CHANNEL_QC_PATH,
    SUMMARY_JSON_PATH,
    SUMMARY_TXT_PATH,
    FRAME_COUNT_PLOT_PATH,
    STAGE_MAP_PLOT_PATH,
]
if WRITE_FILE_MANIFEST:
    output_paths.append(FILE_MANIFEST_PATH)

for output_path in output_paths:
    output_path.parent.mkdir(parents=True, exist_ok=True)

if WRITE_OUTPUTS:
    positions_df.to_csv(POSITION_MANIFEST_PATH, sep="\t", index=False)
    channel_df.to_csv(CHANNEL_QC_PATH, sep="\t", index=False)
    if WRITE_FILE_MANIFEST:
        files_df.to_csv(FILE_MANIFEST_PATH, sep="\t", index=False)
    SUMMARY_JSON_PATH.write_text(json.dumps(dataset_summary, indent=2) + "\n")
    SUMMARY_TXT_PATH.write_text("\n".join(summary_lines) + "\n")

print("Position manifest:", POSITION_MANIFEST_PATH)
print("Channel QC:", CHANNEL_QC_PATH)
if WRITE_FILE_MANIFEST:
    print("File manifest:", FILE_MANIFEST_PATH)


In [ ]:
# -------------------------------
# QC review tables
# -------------------------------
print("Dataset summary")
display(pd.DataFrame([dataset_summary]).T.rename(columns={0: "value"}))

flagged_positions = positions_df.loc[
    positions_df["is_layout_outlier"]
    | positions_df["is_frame_count_outlier"]
    | (~positions_df["matches_metadata_frame_count"])
    | (~positions_df["all_channels_contiguous"])
    | (~positions_df["all_channels_same_frame_count"]),
    [
        "position_label",
        "image_layout",
        "storage_subdirs",
        "avi_count",
        "observed_tiff_count",
        "observed_frames_per_channel",
        "metadata_declared_frames",
        "frame_index_min",
        "frame_index_max",
        "all_channels_same_frame_count",
        "all_channels_contiguous",
        "matches_metadata_frame_count",
    ],
].reset_index(drop=True)

print("Flagged positions")
display(flagged_positions)

print("Per-channel frame counts")
display(
    channel_df.pivot(index="position_label", columns="channel_name", values="observed_frames")
    .sort_index(key=lambda idx: [pos_index_from_name(name) for name in idx])
)

print("Position manifest head")
display(positions_df.head())


In [ ]:
# -------------------------------
# QC plots
# -------------------------------
colors = ["tab:orange" if is_outlier else "tab:blue" for is_outlier in positions_df["is_layout_outlier"]]

fig, ax = plt.subplots(figsize=(15, 4.5))
ax.bar(positions_df["position_label"], positions_df["observed_frames_per_channel"], color=colors)
ax.axhline(
    dataset_summary["metadata_declared_frames"],
    color="crimson",
    linestyle="--",
    linewidth=1.5,
    label=f"metadata declared ({dataset_summary['metadata_declared_frames']})",
)
ax.set_title("Observed frames per position")
ax.set_xlabel("Position")
ax.set_ylabel("Observed frames per channel")
ax.tick_params(axis="x", rotation=90)
ax.legend(loc="upper right")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(FRAME_COUNT_PLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()

stage_df = positions_df.dropna(subset=["stage_x_um", "stage_y_um"]).copy()
fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(
    stage_df["stage_x_um"],
    stage_df["stage_y_um"],
    c=stage_df["observed_frames_per_channel"],
    cmap="viridis",
    s=90,
    edgecolor="black",
    linewidth=0.3,
)
for _, row in stage_df.loc[stage_df["is_layout_outlier"]].iterrows():
    ax.text(row["stage_x_um"], row["stage_y_um"], row["position_label"], fontsize=8)
ax.set_title("Stage map colored by observed frames per channel")
ax.set_xlabel("Stage X (um)")
ax.set_ylabel("Stage Y (um)")
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Observed frames per channel")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(STAGE_MAP_PLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
created_files = [
    POSITION_MANIFEST_PATH,
    CHANNEL_QC_PATH,
    FILE_MANIFEST_PATH if WRITE_FILE_MANIFEST else None,
    SUMMARY_JSON_PATH,
    SUMMARY_TXT_PATH,
    FRAME_COUNT_PLOT_PATH,
    STAGE_MAP_PLOT_PATH,
]

print("Created files")
for path in created_files:
    if path is not None and path.exists():
        print("-", rel_to_root(path))
